# Tutorial 12: Routing CPU work to a Process Pool

The default taskforce in LAILA is `PythonAsyncThreadPoolTaskForce` — perfect for I/O-bound work but bottlenecked by the GIL for CPU-heavy tasks. For genuinely parallel CPU work, register a `PythonProcessPoolTaskForce` and route work to it.

You will:

- Register a second taskforce on the active policy
- Submit a CPU-bound function to the process pool directly via the taskforce
- Compare wall-clock times across both pools
- Tear down both cleanly with `laila.terminate`

**Prerequisites:** `pip install laila-core`.

> **Note** — Running this tutorial in a script (`python my_script.py`) requires the standard `if __name__ == "__main__":` guard around the submission code, because Python's `spawn` start method re-imports the main module in each worker process. Jupyter notebooks side-step this automatically; the cells below run as-is in a notebook.

In [ ]:
import time

import laila
from laila.policy.central.command.taskforce.process_pool_executor.taskforce import (
    PythonProcessPoolTaskForce,
)

print("alpha taskforce:", laila.command.alpha_taskforce[:16], "...")
print("registered:", list(laila.command.taskforces.keys()))

## Register the process pool

`PythonProcessPoolTaskForce` requires at least four workers (the model enforces `num_workers >= 4`). `add_taskforce` registers it under its own `global_id`.

In [ ]:
proc_tf = PythonProcessPoolTaskForce(
    num_workers=4,
    policy_id=laila.active_policy.global_id,
)
laila.command.add_taskforce(proc_tf)
proc_tf.start()

print("process taskforce gid:", proc_tf.global_id)
print("now registered:", len(laila.command.taskforces), "taskforces")

## A CPU-bound workload

Computing `math.factorial(120000)` is GIL-bound — async threads cannot parallelise it because the work happens inside a single interpreter. Process workers run in separate Python interpreters so they each get their own GIL.

We use `math.factorial` rather than a notebook-defined function because process pool workers must be able to **import** the callable they receive. Module-level functions in real Python files, builtins, and `functools.partial` over those all work; notebook-defined `def` functions and lambdas do not (the workers cannot resolve them by name).

In [ ]:
from math import factorial
from functools import partial

# A picklable, importable callable: math.factorial wrapped with a fixed arg.
task = partial(factorial, 200_000)

## Submit on the alpha (async-thread) taskforce

Without a `taskforce_id`, `laila.command.submit` routes to `alpha_taskforce`. Four tasks run in parallel threads — but each is CPU-bound, so the GIL serialises them.

The async-thread pool accepts lambdas and other closures because tasks run in-process. We use `functools.partial` here as a forward-compatible substitute since the process pool below will require picklable callables.

In [ ]:
t0 = time.perf_counter()
fut = laila.command.submit([task for _ in range(4)])
fut.wait()
print("alpha pool elapsed:", round(time.perf_counter() - t0, 2), "s")

## Submit on the process pool

`laila.command.submit` wraps tasks with a coroutine-adapter closure that is not picklable, so for process-pool submission you call the taskforce's `submit` method directly. The argument shape is the same:

In [ ]:
t0 = time.perf_counter()
group = proc_tf.submit([task for _ in range(4)], wait=False)
group.wait()
print("process pool elapsed:", round(time.perf_counter() - t0, 2), "s")

## Tear down both cleanly

`laila.terminate(wait=True)` drains every registered taskforce in turn. Pass `cancel_pending=True` to drop queued-but-unstarted submissions instead of waiting them out.

In [ ]:
errors = laila.terminate(wait=True)
print("teardown errors:", errors)

## When to use a process pool

| Workload | Taskforce |
|---|---|
| I/O, network, async serialization | Default async-thread pool |
| Pure-Python CPU work (hashing, parsing, image processing) | Process pool |
| Native extension calls that release the GIL (numpy, torch math) | Either — async-thread is often enough |
| Crash-prone code that should not take down the policy | Process pool (worker isolation) |

## Summary

- `PythonProcessPoolTaskForce(num_workers=N)` requires `N >= 4`.
- `laila.command.add_taskforce(tf)` registers it on the active policy.
- For the async-thread default, call `laila.command.submit([fn, ...])`.
- For the process pool, call `tf.submit([fn, ...], wait=False)` directly because `laila.command.submit` wraps tasks in a closure that processes cannot pickle.
- Tasks routed to a process pool must be **top-level picklable** callables — module-level functions or `functools.partial(fn, *args)`. Lambdas will not pickle.
- `laila.terminate(wait=True)` cleans up every pool on the active policy.